# RSNA knee training runner

Run all. It reads `jobs.txt` (`jobs_A.txt` or `jobs_B.txt` in the runner_A and runner_B copies) and `cache_kernels.txt` from the Drive folder `rsna_knee`, downloads the cache from Kaggle once per session, trains every job that has no `runs/<name>/DONE`, and writes logs and checkpoints back to Drive.

One-time setup. Add a Colab secret named `KAGGLE_API_TOKEN` (key icon on the left) holding the Kaggle API token, and enable notebook access for it.

When the jobs run out the last cell releases the runtime. Set `AUTO_STOP = False` in the first code cell to keep it.

If the runtime disconnects, run all again. The unfinished job resumes after its last completed epoch.

In [ ]:
import os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/rsna_knee')
# tools/sync_colab.sh also writes runner_A.ipynb and runner_B.ipynb with jobs_A.txt and jobs_B.txt here, so two sessions never pick the same job.
JOBS_FILE = 'jobs.txt'
# Release the runtime once the jobs file is exhausted, so a run left overnight stops spending compute units.
AUTO_STOP = True
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
(kaggle_dir / 'access_token').write_text(userdata.get('KAGGLE_API_TOKEN').strip())
(kaggle_dir / 'access_token').chmod(0o600)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'kaggle', 'timm'], check=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout)
print('cpus', os.cpu_count())

In [ ]:
import re, time, zipfile
import requests

CACHE = Path('/content/cache')
lines = [l.split() for l in (ROOT / 'cache_kernels.txt').read_text().splitlines() if l.strip() and not l.startswith('#')]
SLOTS = next(l[1] for l in lines if l[0] == 'slots')
# `env KEY=VALUE` lines carry the other build settings, which train.py checks against meta.csv.
CACHE_ENV = dict(l[1].split('=', 1) for l in lines if l[0] == 'env')
kernels = [l[0] for l in lines if l[0] not in ('slots', 'env')]


def urls(kernel):
    out = subprocess.run([sys.executable, str(ROOT / 'code' / 'kernel_output_urls.py'), f'zacheenoao/{kernel}', r'cache_train.*\.zip'],
                         capture_output=True, text=True, check=True).stdout
    return dict(l.split(' ', 1) for l in out.strip().splitlines())


def fetch(kernel, name):
    """Resume until the size matches, signing a fresh URL per attempt (the old one may have expired), because the server drops long transfers."""
    f = Path('/content') / name
    total = int(requests.get(urls(kernel)[name], headers={'Range': 'bytes=0-0'}).headers['Content-Range'].split('/')[1])
    while (have := f.stat().st_size if f.exists() else 0) < total:
        print(f'{time.strftime("%H:%M")} {name} {have >> 20} of {total >> 20} MB', flush=True)
        try:
            with requests.get(urls(kernel)[name], headers={'Range': f'bytes={have}-'}, stream=True, timeout=60) as r, f.open('ab') as out:
                r.raise_for_status()
                for chunk in r.iter_content(1 << 22):
                    out.write(chunk)
        except Exception as e:
            print('retry after', type(e).__name__, e)
            time.sleep(10)
    return f


for k in kernels:
    names = urls(k)
    if not names:  # a build still running has no zip yet, and training on part of the cache would go unnoticed
        raise SystemExit(f'{k} has no cache zip yet, wait until it finishes')
    for name in names:
        tag = re.sub(r'^cache_|\.zip$', '', name)
        if (CACHE / tag / 'meta.csv').exists():
            continue
        z = fetch(k, name)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(CACHE)
        z.unlink()
print(sorted(p.name for p in CACHE.iterdir()), 'slots', SLOTS, CACHE_ENV)

In [ ]:
env = {**os.environ, **CACHE_ENV, 'RSNA_SLOTS': SLOTS, 'RSNA_DATA_ROOT': str(ROOT / 'raw'), 'RSNA_WORK': '/content/work',
       'PYTHONIOENCODING': 'utf-8'}
workers = str(min(8, os.cpu_count()))

# The jobs file is re-read after every job, so jobs added on Drive while this runs are picked up.
# Checkpoints and resume_state.pt go straight to the Drive run folder every epoch, so rerunning after a dropped
# runtime continues the unfinished job from its last epoch. train.log is appended to for the same reason.
while True:
    jobs = [l.split() for l in (ROOT / JOBS_FILE).read_text().splitlines() if l.strip() and not l.startswith('#')]
    todo = [j for j in jobs if not (ROOT / 'runs' / j[0] / 'DONE').exists()]
    if not todo:
        print('no jobs left')
        break
    name, labels, seed, *extra = todo[0]
    run = ROOT / 'runs' / name
    run.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, '-u', str(ROOT / 'code' / 'train.py'), '--cache', str(CACHE / 'train_*'),
           '--labels', str(ROOT / 'labels' / labels), '--epochs', '12', '--bs', '4', '--workers', workers, '--seed', seed,
           '--lr', '3e-4', '--backbone-lr', '3e-4', '--wd', '1e-4', '--dropout', '0', '--out', str(run / 'kneenet.pt'), '--resume', *extra]
    print(time.strftime('%H:%M'), 'start', name, ' '.join(cmd[3:]), flush=True)
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
        # Reopen per line, because the Drive mount uploads a file only once it is closed.
        with (run / 'train.log').open('a') as log:
            log.write(line)
    if p.wait() != 0:
        print(time.strftime('%H:%M'), name, 'failed, see', run / 'train.log')
        break
    (run / 'DONE').write_text(time.strftime('%Y-%m-%d %H:%M'))
    print(time.strftime('%H:%M'), 'done', name, flush=True)

In [ ]:
# Run all reaches this cell only after the job loop above ends, whether every job finished or one failed.
if AUTO_STOP:
    from google.colab import runtime
    runtime.unassign()
